# gambit_game: rules, engine, physical moves and the game loop

This package turns "the camera saw a board" into "the robot made a legal reply on the physical
board, and everything was logged". It sits between `gambit_vision` (board + per-square
confidence) and `gambit_arm` (pick/place primitives), and asks `gambit_teach` which move to play.

## Deliverables

| Deliverable | Notebook | Status |
|---|---|---|
| Rules / legality (python-chess; legality checked separately from placement) | `game_manager.ipynb` (`match_legal_moves`, VALIDATE) | TODO (skeleton) |
| Engine at a chosen strength (Stockfish UCI + fallbacks) | `engine.ipynb` | TODO (skeleton), Elo calibration TODO |
| Physical move execution for every special move | `physical_moves.ipynb` | TODO (skeleton) |
| Game loop, camera as source of truth | `game_manager.ipynb`, `game_loop.ipynb` | state enum + transition table given; rest TODO |
| Mismatch detection and recovery | `recovery.ipynb` | TODO (skeleton) |
| Think-time / hesitation features | `hesitation.ipynb` | data structures given; rest TODO |
| Logging (PGN + JSONL) | `logger.ipynb` | TODO (skeleton) |
| Real game on robot / Gazebo | `run_play_game.ipynb` | TODO adapters |
| Engine strength calibration (~1700) | `run_engine_strength.ipynb` | TODO |

## Interfaces assumed from other packages

* **Vision** -> `BoardObservation(placement, confidence[64], timestamp, hand_present)`
  (`recovery.ipynb`). `placement` is the FEN piece-placement field.
* **Arm** <- `ArmInterface` protocol (`physical_moves.ipynb`): `pick(sq)`, `place(sq)`,
  `move_piece(from, to)`, `capture_to_graveyard(sq)`, `place_from_graveyard(piece, sq)`, `park()`,
  each returning `True` on success. Squares are python-chess indices.
* **Teacher** <- `MovePolicy.choose_move(board)` (`game_manager.ipynb`); implemented by
  `gambit_teach/notebooks/move_selection.ipynb`.
* Config: `gambit_bringup/config/game.yaml` (engine, thresholds, timeouts, logging) and
  `board.yaml` (`spare_queens`, `graveyard`).

## Run order

1. `engine.ipynb`, `physical_moves.ipynb`, `recovery.ipynb` (independent building blocks)
2. `game_manager.ipynb` (uses the three above)
3. `logger.ipynb`, `hesitation.ipynb`
4. `game_loop.ipynb` (everything wired; its self-check plan plays a short software-only game)
5. `run_engine_strength.ipynb`, then `run_play_game.ipynb`

```bash
tools/run_notebooks.sh gambit_game     # executes every library notebook
```

Every library notebook is a skeleton: signatures, data types and design notes are given, the
bodies raise `NotImplementedError`, and each self-check cell lists the tests to write.

## TODO checklist (package level)
- [ ] Implement the library notebooks in the run order above, writing each self-check as you go.
- [ ] Install Stockfish and check its license (README "License").
- [ ] Recovery policy + human prompts (`recovery.ipynb`, `GameManager.ask_human`).
- [ ] Vision and arm adapters for `run_play_game.ipynb`.
- [ ] Calibrate engine strength (`run_engine_strength.ipynb`).
- [ ] Measure: per-move success, retries, full games without intervention (README milestones).